In [1]:
import re
import pandas as pd
import numpy as np
import nltk
import pymorphy3
import optuna
import mlflow
import pickle
import implicit

from implicit.als import AlternatingLeastSquares
from scipy.sparse import coo_matrix, csr_matrix, vstack
from optuna.integration.mlflow import MLflowCallback
from mlflow.utils.mlflow_tags import MLFLOW_PARENT_RUN_ID
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import ndcg_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from nltk.corpus import stopwords
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from gensim.utils import simple_preprocess
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import warnings
warnings.simplefilter('ignore', FutureWarning)

from sklearn import set_config
set_config(display='diagram')

In [2]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [3]:
TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

EXPERIMENT_NAME = "hr-ai-scout"
STUDY_DB_NAME = "sqlite:///local.study.db"

mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

# Load data

In [4]:
df = pd.read_csv('/Users/user/Documents/Magistracy/year_project/hr-ai-scout/total_df.csv')
df.head()

,vacancy_id,vacancy_name,vacancy_area,vacancy_experience,vacancy_employment,vacancy_schedule,vacancy_salary_from,vacancy_salary_to,vacancy_salary_currency,vacancy_salary_gross,...,resume_education,resume_courses,resume_salary,resume_age,resume_total_experience,resume_experience_months,resume_location,resume_gender,resume_applicant_status,target
0,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,300000.0,NaN,RUR,False,...,['Казанский Авиационный Институт'],NaN,NaN,65.0,19 лет,228.0,Москва,Мужчина,Рассматривает предложения,1
1,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,300000.0,NaN,RUR,False,...,"['ООО ""Открытый Учебный Центр СофтБаланс"", г. ...","['ООО ""Открытый Учебный Центр СофтБаланс"", г. ...",NaN,43.0,17 лет 4 месяца,208.0,Москва,Мужчина,Рассматривает предложения,1
2,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,300000.0,NaN,RUR,False,...,['Орский государственный педагогический инстит...,NaN,200 000 ₽ на руки,52.0,30 лет,360.0,Москва,Женщина,NaN,1
3,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,300000.0,NaN,RUR,False,...,['Красноярский государственный университет'],NaN,500 000 ₽ на руки,56.0,29 лет 8 месяцев,356.0,Красноярск,Мужчина,Рассматривает предложения,1
4,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,300000.0,NaN,RUR,False,...,['Белоруский Гос. Университет Информатики и Ра...,"['SAP CIS, SAP XI', 'Школа Логистики МАДИ', 'S...",NaN,48.0,25 лет 1 месяц,301.0,Moscow,Male,NaN,1


# Preprocessing

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">
В первую очередь уберем строки, где пропущены все ключевые значения в резюме:
</div>

In [5]:
t1 = df.shape[0]
df = df.dropna(subset= ["resume_education",
                        "resume_last_experience_description",
                        "resume_last_position",
                        "resume_last_company_experience_period",
                        "resume_total_experience",
                        "resume_experience_months",
                        "resume_location",
                        "resume_specialization",
                        # "resume_gender",
                        # "resume_title"
                       ], how="all")
t2 = df.shape[0]
print('Удалено ', t1 - t2 ,' строки')

Удалено  84  строки


<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">
Удалим еще те строки, где случился технический сбой в парсинге, где у кандидата общий опыт есть, а последний опыт не указан (и наоборот):
</div>

In [6]:
t1 = df.shape[0]
df = df.loc[~(df["resume_total_experience"].notna()
        & df["resume_last_experience_description"].isna()
        & df["resume_last_position"].isna())]
t2 = df.shape[0]
print('Удалено ', t1 - t2 ,' строк')

Удалено  1543  строк


In [7]:
t1 = df.shape[0]
df = df.loc[~(df["resume_total_experience"].isna()
        & df["resume_last_experience_description"].notna()
        & df["resume_last_position"].notna())]
t2 = df.shape[0]
print('Удалено ', t1 - t2 ,' строк')

Удалено  0  строк


<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">
Посмотрим на пропуски отдельно по категориальным и числовым признакам.
</div>

In [8]:
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(include=['object']).columns

In [9]:
df[cat_cols] = df[cat_cols].fillna('NDT')

In [10]:
df.loc[df['resume_experience_months'].isna(), 'resume_last_experience_description'].unique()

array(['NDT'], dtype=object)

In [11]:
df['resume_age'] = df['resume_age'].fillna(df['resume_age'].mean())
df['resume_experience_months'] = df['resume_experience_months'].fillna(0)

In [12]:
df = df.drop(['vacancy_salary_to', 'vacancy_salary_from',
              'vacancy_salary_currency', 'vacancy_salary_gross'], axis=1)

In [13]:
df.loc[df['resume_last_company_experience_period'] == 'NDT', 'resume_last_experience_description'].unique()

array(['NDT'], dtype=object)

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">
Преобразуем сначала ожидаемые зарплаты
</div>

In [14]:
df['resume_salary_split'] = df['resume_salary'].apply(lambda x: x.split())

df['salary_int'] = df['resume_salary_split'].apply(
    lambda x: int(''.join(part for part in x if re.fullmatch(r'\d+', part)))
              if any(re.fullmatch(r'\d+', part) for part in x)
              else np.nan
)

currency_symbols = ['₽', '$', '€', '₴', '₸', '₼', '₾', 'Br', "so'm"]

rates_rub = {
    "₽": 1.0,
    "$": 80.85,
    "€": 94.14,
    "₴": 1.94,
    "₸": 0.150,
    "₼": 47.8,
    "₾": 33.5,
    "Br": 28.7,
    "so'm": 0.0068
}

df['currency_symbol'] = df['resume_salary_split'].apply(
    lambda x: next((sym for sym in x if sym in currency_symbols), np.nan)
)

df['salary_converted'] = (df['salary_int'] * df['currency_symbol'].map(rates_rub)).fillna(0)

df['resume_salary'] = df['salary_converted']

df = df.drop(['resume_salary_split', 'salary_int', 'currency_symbol', 'salary_converted'], axis=1)

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">
Добавим дополнительный столбец с опытом работы в последней компании в месяцах для удобства
</div>

In [15]:
def experience_to_months(experience_text):
    months = 0
    # Опыт в годах
    years_match = re.search(r'(\d+)\s*год', experience_text)
    if years_match:
        months += int(years_match.group(1)) * 12

    years_match = re.search(r'(\d+)\s*лет', experience_text)
    if years_match:
        months += int(years_match.group(1)) * 12

    # Опыт в месяцах
    months_match = re.search(r'(\d+)\s*месяц', experience_text)
    if months_match:
        months += int(months_match.group(1))

    return months if months > 0 else np.nan

In [16]:
df['resume_last_company_experience_months'] = df['resume_last_company_experience_period'].apply(experience_to_months)

In [17]:
df.loc[df['resume_last_company_experience_period'] == 'NDT', 'resume_last_experience_description'].unique()

array(['NDT'], dtype=object)

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">
Т.к. в названии компании стоит NDT, можно столбец resume_last_company_experience_months заполнять нулевыми значениями.
</div>

In [18]:
df['resume_last_company_experience_months'] = df['resume_last_company_experience_months'].fillna(0)

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

- Ограничим выбросы по зарплате, потому что ровно одно значение по ожидаемой заработоной плате = 999,999,999 (смешно, но нет)

- Ограничим опыт общий и внутри одной компании до 720 месяцев (60 лет, ничего себе уже)

- Уберем возраст > 90, не ждем, что эти кандидаты находятся в поиске вакансии
</div>

In [19]:
df = df[~(df.resume_salary > 1e7)]
df.loc[df['resume_experience_months'] > 720, 'resume_experience_months'] = 720
df.loc[df['resume_last_company_experience_months'] > 720, 'resume_last_company_experience_months'] = 720
df = df[~(df.resume_age > 90)]

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

- Также уберем строки, где последний опыт кандидата больше, чем общий

- И где общий опыт кандидата +16 лет больше чем возраст (хоть так)

</div>

In [20]:
df = df[~(df.resume_experience_months < df.resume_last_company_experience_months)]
df = df[~(df.resume_age < (df.resume_experience_months // 12) + 16)]

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Заменим текущий формат разброса полов в датасете на унифицированный

</div>

In [21]:
gender_map = {
    'Мужчина': 'Мужчина',
    'Male': 'Мужчина',
    'Женщина': 'Женщина',
    'Female': 'Женщина'
}

df['resume_gender'] = df['resume_gender'].apply(lambda x: gender_map[x] if x in gender_map else 'Неизвестно')

In [22]:
print(f"Датасет после очистки: {df.shape}")

Датасет после очистки: (325543, 25)


# Feature engineering

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">
Добавим признак матчинга локации вакансии и резюме

</div>

In [23]:
df['location_matching'] = df.apply(lambda row: 1 if row['vacancy_area'] == row['resume_location'] else 0, axis=1)

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Сделаем новый признак, а именно посчитаем количество навыков кандидата, которые указаны в вакансии.

</div>

In [24]:
def resume_skill_count_in_vacancy(row):
    count = 0
    skill_list = row['resume_skills'].replace('[', '').replace(']', '').replace("'", "").split(', ')
    for i in skill_list:
        if i in row['vacancy_description']:
            count += 1
    return count

df['resume_skill_count_in_vacancy'] = df.apply(resume_skill_count_in_vacancy, axis=1)

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Также посчитаем долю слов из последней должности в резюме, которые указаны в вакансии.

</div>

In [25]:
def last_position_in_vacancy(row):
    bow = []
    seps = [' ', '-', '_']
    for sep in seps:
        bow += row['resume_last_position'].split(sep=sep)
        bow = list(set(bow))
    
    c = 0
    for word in bow:
        if word in row['vacancy_description']:
            c +=1
    
    return c / len(bow)

In [26]:
df['last_position_in_vacancy'] = df.apply(last_position_in_vacancy, axis=1)

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Теперь закодируем описание вакансии и последнего опыта работы и сравним через косинусное расстояние.

</div>

In [27]:
def preprocess_data(df):
    """Обработка пропущенных значений в текстовых полях"""
    print("Проверка пропущенных значений...")
    print(f"Пропуски в vacancy_description: {df['vacancy_description'].isna().sum()}")
    print(f"Пропуски в resume_last_experience_description: {df['resume_last_experience_description'].isna().sum()}")
    
    # Заполняем пропуски пустыми строками
    df['vacancy_description'] = df['vacancy_description'].fillna('')
    df['resume_last_experience_description'] = df['resume_last_experience_description'].fillna('')
    
    # Проверяем, что все значения теперь строковые
    df['vacancy_description'] = df['vacancy_description'].astype(str)
    df['resume_last_experience_description'] = df['resume_last_experience_description'].astype(str)
    
    return df

In [28]:
def save_results(df, output_file):
    """Сохранение результатов в CSV файл"""
    df.to_csv(output_file, index=False, encoding='utf-8')
    print(f"Результаты сохранены в файл: {output_file}")

In [29]:
def calculate_cosine_similarity(embeddings1, embeddings2):
    """Вычисление косинусного сходства между двумя наборами эмбеддингов"""
    similarities = []
    
    for i in tqdm(range(embeddings1.shape[0])):
        emb1_row = embeddings1[i]
        emb2_row = embeddings2[i]
        
        similarity = cosine_similarity(emb1_row, emb2_row)[0][0]
        similarities.append(similarity)
    
    return similarities

In [30]:
warnings.filterwarnings('ignore')

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

try:
    nltk.data.find('taggers/averaged_perceptron_tagger_ru')
except LookupError:
    nltk.download('averaged_perceptron_tagger_ru')

try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')

morph = pymorphy3.MorphAnalyzer()

[nltk_data] Downloading package wordnet to /Users/user/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [31]:
def lemmatize_russian(tokens):
    """Лемматизация русских слов"""
    lemmas = []
    for token in tokens:
        parsed = morph.parse(token)[0]  # Берем самый вероятный разбор
        lemmas.append(parsed.normal_form)
    return lemmas

In [32]:
def tokenize_and_lemmatize(text):
    """Токенизация текста с лемматизацией и удалением стоп-слов"""
    tokens = simple_preprocess(text, deacc=True, min_len=2)
    stop_words = set(stopwords.words('russian') + stopwords.words('english'))
    tokens = [token for token in tokens if token not in stop_words]
    lemmatized_tokens = lemmatize_russian(tokens)
    
    return lemmatized_tokens

In [33]:
def get_tfidf_embeddings(texts, vectorizer=None, fit=True):
    """Создание TF-IDF эмбеддингов для списка текстов с лемматизацией"""
    if fit:
        vectorizer = TfidfVectorizer(
            max_features=5000,
            min_df=2,
            max_df=0.8,
            ngram_range=(1, 2),
            tokenizer=tokenize_and_lemmatize,
            token_pattern=None,
            lowercase=False  # Уже сделано в токенизации
        )
        embeddings = vectorizer.fit_transform(texts)
    else:
        embeddings = vectorizer.transform(texts)
    
    return embeddings, vectorizer

In [34]:
def get_tfidf_vacancy_embeddings(df, vectorizer=None):
    """Создание эмбеддингов для уникальных вакансий с лемматизацией"""
    unique_vacancies = df[['vacancy_id', 'vacancy_description']].drop_duplicates()
    
    unique_embeddings, vectorizer = get_tfidf_embeddings(
        unique_vacancies['vacancy_description'].tolist(), 
        vectorizer=vectorizer, 
        fit=(vectorizer is None)
    )
    
    vacancy_embedding_dict = dict(zip(unique_vacancies['vacancy_id'], unique_embeddings))
    
    rows = []
    for vid in df['vacancy_id']:
        rows.append(vacancy_embedding_dict[vid])
    
    all_vacancy_embeddings = vstack(rows)
    
    return all_vacancy_embeddings, vectorizer

In [35]:
def process_similarity_scores_tfidf(df, vectorizer=None, fit=True):
    """Функция для вычисления схожести с использованием TF-IDF и лемматизации"""
    df = preprocess_data(df)
    
    print("Создание TF-IDF эмбеддингов для описаний опыта в резюме...")
    experience_embeddings, tfidf_vectorizer = get_tfidf_embeddings(df['resume_last_experience_description'].tolist(), vectorizer=vectorizer, fit=fit)
    
    print("Создание TF-IDF эмбеддингов для описаний вакансий...")
    vacancy_embeddings, _ = get_tfidf_vacancy_embeddings(df, vectorizer=tfidf_vectorizer)
    
    print("Вычисление косинусного сходства...")
    similarity_scores = calculate_cosine_similarity(vacancy_embeddings, experience_embeddings)
    
    df['similarity_score_tfidf'] = similarity_scores
    
    return df, tfidf_vectorizer

In [36]:
try:
    df_tfidf = pd.read_csv('description_df_with_scores_tfidf.csv')
except:
    df_tfidf = process_similarity_scores_tfidf(df.copy())
    save_results(df_tfidf, 'description_df_with_scores_tfidf.csv')

In [37]:
df = df.merge(df_tfidf)

In [38]:
df.head()

,vacancy_id,vacancy_name,vacancy_area,vacancy_experience,vacancy_employment,vacancy_schedule,vacancy_description,resume_id,resume_title,resume_specialization,...,resume_location,resume_gender,resume_applicant_status,target,resume_last_company_experience_months,location_matching,resume_skill_count_in_vacancy,last_position_in_vacancy,resume_skill_count,similarity_score_tfidf
0,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,"Привет!.redev — технологическая компания, созд...",6969174,ABAP-разработчик,"['Программист, разработчик']",...,Москва,Мужчина,Рассматривает предложения,1,76.0,1,3,0.666667,3,0.284047
1,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,"Привет!.redev — технологическая компания, созд...",9100077,"ABAP разработчик - SAP HCM, CRM, S/4HANA ERP(F...","['Программист, разработчик']",...,Москва,Мужчина,Рассматривает предложения,1,8.0,1,2,0.500000,2,0.308726
2,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,"Привет!.redev — технологическая компания, созд...",32644957,Разработчик ABAP,"['Программист, разработчик']",...,Москва,Женщина,NDT,1,136.0,1,1,0.000000,1,0.510093
3,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,"Привет!.redev — технологическая компания, созд...",27220466,ABAP-разработчик,"['Программист, разработчик']",...,Красноярск,Мужчина,Рассматривает предложения,1,135.0,0,2,0.333333,2,0.301062
4,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,"Привет!.redev — технологическая компания, созд...",7532708,ABAP разработчик. Senior ABAP Developer. SAP T...,"['Programmer, developer']",...,Moscow,Мужчина,NDT,1,0.0,0,2,0.600000,2,0.075429


# Train-test split

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Добавим новые признаки в обучение и тестирование

</div>

In [39]:
features = [
    'vacancy_area',
    'vacancy_experience',
    'vacancy_employment', 
    'vacancy_schedule',
    # 'resume_specialization',
    # 'resume_education', 
    # 'resume_courses', 
    'resume_salary',
    'resume_age', 
    'resume_experience_months',
    'resume_location',
    'resume_gender', 
    'resume_applicant_status', 
    'resume_last_company_experience_months', 
    'location_matching',
    'resume_skill_count_in_vacancy',
    'last_position_in_vacancy',
    'similarity_score_tfidf'
]
df[features]

,vacancy_area,vacancy_experience,vacancy_employment,vacancy_schedule,resume_salary,resume_age,resume_experience_months,resume_location,resume_gender,resume_applicant_status,resume_last_company_experience_months,location_matching,resume_skill_count_in_vacancy,last_position_in_vacancy,similarity_score_tfidf
0,Москва,Более 6 лет,Полная занятость,Удаленная работа,0.0,65.000000,228.0,Москва,Мужчина,Рассматривает предложения,76.0,1,3,0.666667,0.284047
1,Москва,Более 6 лет,Полная занятость,Удаленная работа,0.0,43.000000,208.0,Москва,Мужчина,Рассматривает предложения,8.0,1,2,0.500000,0.308726
2,Москва,Более 6 лет,Полная занятость,Удаленная работа,200000.0,52.000000,360.0,Москва,Женщина,NDT,136.0,1,1,0.000000,0.510093
3,Москва,Более 6 лет,Полная занятость,Удаленная работа,500000.0,56.000000,356.0,Красноярск,Мужчина,Рассматривает предложения,135.0,0,2,0.333333,0.301062
4,Москва,Более 6 лет,Полная занятость,Удаленная работа,0.0,48.000000,301.0,Moscow,Мужчина,NDT,0.0,0,2,0.600000,0.075429
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
321637,Москва,От 3 до 6 лет,Полная занятость,Удаленная работа,242550.0,66.000000,521.0,Санкт-Петербург,Женщина,NDT,270.0,0,0,0.166667,0.072670
321638,Москва,От 3 до 6 лет,Полная занятость,Удаленная работа,0.0,40.000000,213.0,Москва,Мужчина,Активно ищет работу,35.0,1,0,0.000000,0.000000
321639,Москва,От 3 до 6 лет,Полная занятость,Удаленная работа,80000.0,44.060813,121.0,Москва,Мужчина,NDT,44.0,1,0,0.200000,0.047398
321640,Москва,От 3 до 6 лет,Полная занятость,Удаленная работа,0.0,32.000000,117.0,Москва,Женщина,NDT,96.0,1,0,0.200000,0.029086


In [40]:
numeric_features = df[features].select_dtypes(include=np.number).columns
categorical_features = df[features].select_dtypes(exclude=np.number).columns

In [41]:
X = df[features].copy()
y = df['target'].copy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

In [42]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((257313, 15), (64329, 15), (257313,), (64329,))

In [43]:
def calculate_metrics(df_test: pd.DataFrame) -> pd.DataFrame:
    ndcg_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    vacancy_ids = df_test['vacancy_id'].unique()
    
    for vacancy_id in vacancy_ids:
        mask = df_test['vacancy_id'] == vacancy_id
        y_true = df_test.loc[mask, 'target'].values
        y_score = df_test.loc[mask, 'y_pred_proba'].values
        
        if len(y_true) <= 1:
            continue
        
        y_true_2d = y_true.reshape(1, -1)
        y_score_2d = y_score.reshape(1, -1)
        
        ndcg = ndcg_score(y_true_2d, y_score_2d)
        ndcg_scores.append(ndcg)
        
        y_pred_binary = (y_score >= 0.5).astype(int)
        
        precision = precision_score(y_true, y_pred_binary, zero_division=0)
        recall = recall_score(y_true, y_pred_binary, zero_division=0)
        f1 = f1_score(y_true, y_pred_binary, zero_division=0)
        
        precision_scores.append(precision)
        recall_scores.append(recall)
        f1_scores.append(f1)
    
    if ndcg_scores:
        print(f"Средний NDCG: {np.mean(ndcg_scores):.4f}")
        print(f"Средний Precision: {np.mean(precision_scores):.4f}")
        print(f"Средний Recall: {np.mean(recall_scores):.4f}")
        print(f"Средний F1-Score: {np.mean(f1_scores):.4f}")

        return np.mean(ndcg_scores), np.mean(precision_scores), np.mean(recall_scores), np.mean(f1_scores)
    else:
        print("Недостаточно данных для расчета метрик")

        return None, None, None, None

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Обучим с подбором гиперпараметров `LogisticRegression`, как бейзлайн для сравнения с нелинейными моделями, `RandomForestClassifier`, как разновидность бэггинга, и три разные вида градиентного бустинга: `XGBClassifier`, `LGBMClassifier` и `CatBoostClassifier`.

</div>

# LogisticRegression

In [44]:
def objective(trial: optuna.Trial) -> float:
    params = {
        'model__C': trial.suggest_float('C', 1e-4, 1e4, log=True),
        'model__penalty': trial.suggest_categorical('penalty', ['l1', 'l2']),
        'model__solver': trial.suggest_categorical('solver', ['liblinear', 'saga'])
    }
    
    pipeline_lr_optuna = Pipeline([
        ('preprocessing', ColumnTransformer([
            ('numeric_scaling', StandardScaler(), numeric_features),
            ('categorical_encoding', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
        ])),
        ('model', LogisticRegression(random_state=RANDOM_STATE, class_weight='balanced'))
    ])
    
    pipeline_lr_optuna.set_params(**params)
    
    kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    
    for train_idx, val_idx in kfold.split(X_train, y_train):
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        pipeline_lr_optuna.fit(X_fold_train, y_fold_train)
        y_pred_proba = pipeline_lr_optuna.predict_proba(X_fold_val)
        

        df_val = df.loc[X_fold_val.index].copy()
        df_val['y_pred_proba'] = y_pred_proba[:, 1]
        
        ndcg, _, _, _ = calculate_metrics(df_val)

        trial.set_user_attr('pipeline_params', params)
    
    return ndcg

In [45]:
try:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
except:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)

RUN_NAME_OPTUNE = 'LogisticRegression_optuna'

with mlflow.start_run(run_name=RUN_NAME_OPTUNE, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

🏃 View run LogisticRegression_optuna at: http://127.0.0.1:5000/#/experiments/1/runs/f89e70c8bbc04aa4bdfd3b3a81a7baa0
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [46]:
STUDY_NAME = "LogisticRegression_optuna"

mlflc = MLflowCallback(
    tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
    metric_name="NDCG",
    create_experiment=False,
    mlflow_kwargs={'experiment_id': experiment_id, 'tags': {MLFLOW_PARENT_RUN_ID: run_id}}
)

In [47]:
study = optuna.create_study(direction='maximize', 
                            sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                            study_name=STUDY_NAME,
                            storage=STUDY_DB_NAME,
                            load_if_exists=False)
study.optimize(objective, 
               n_trials=20, 
               callbacks=[mlflc]
              )
best_params_optuna = study.best_params

print(f"Number of finished trials: {len(study.trials)}")
print(f"Best params: {best_params_optuna}")

[I 2026-05-08 16:11:30,917] A new study created in RDB with name: LogisticRegression_optuna


Средний NDCG: 0.8300
Средний Precision: 0.5702
Средний Recall: 0.7888
Средний F1-Score: 0.6312
Средний NDCG: 0.8191
Средний Precision: 0.5614
Средний Recall: 0.7788
Средний F1-Score: 0.6213


[I 2026-05-08 16:12:08,649] Trial 0 finished with value: 0.8270779138714848 and parameters: {'C': 0.09915644566638405, 'penalty': 'l1', 'solver': 'liblinear'}. Best is trial 0 with value: 0.8270779138714848.


Средний NDCG: 0.8271
Средний Precision: 0.5711
Средний Recall: 0.7823
Средний F1-Score: 0.6304
🏃 View run 0 at: http://127.0.0.1:5000/#/experiments/1/runs/d8f88ea51cf94f2081fdc9f99fe80a45
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8260
Средний Precision: 0.5665
Средний Recall: 0.7821
Средний F1-Score: 0.6264
Средний NDCG: 0.8150
Средний Precision: 0.5597
Средний Recall: 0.7711
Средний F1-Score: 0.6166


[I 2026-05-08 16:12:27,362] Trial 1 finished with value: 0.8219108201868665 and parameters: {'C': 0.001769930294063334, 'penalty': 'l2', 'solver': 'saga'}. Best is trial 0 with value: 0.8270779138714848.


...

Средний NDCG: 0.8269
Средний Precision: 0.5715
Средний Recall: 0.7817
Средний F1-Score: 0.6303
🏃 View run 17 at: http://127.0.0.1:5000/#/experiments/1/runs/618c699994b54e92846e68b87280f57e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8302
Средний Precision: 0.5693
Средний Recall: 0.7897
Средний F1-Score: 0.6310
Средний NDCG: 0.8192
Средний Precision: 0.5623
Средний Recall: 0.7797
Средний F1-Score: 0.6222


[I 2026-05-08 16:24:08,464] Trial 18 finished with value: 0.8271089575233356 and parameters: {'C': 0.940927363855838, 'penalty': 'l1', 'solver': 'saga'}. Best is trial 11 with value: 0.8271110148455479.


Средний NDCG: 0.8271
Средний Precision: 0.5718
Средний Recall: 0.7821
Средний F1-Score: 0.6307
🏃 View run 18 at: http://127.0.0.1:5000/#/experiments/1/runs/c5abb44aa621467e87a9794f9df37a9b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8304
Средний Precision: 0.5688
Средний Recall: 0.7896
Средний F1-Score: 0.6307
Средний NDCG: 0.8191
Средний Precision: 0.5618
Средний Recall: 0.7795
Средний F1-Score: 0.6220


[I 2026-05-08 16:24:27,686] Trial 19 finished with value: 0.8269458488463799 and parameters: {'C': 49.638930718338166, 'penalty': 'l2', 'solver': 'saga'}. Best is trial 11 with value: 0.8271110148455479.


Средний NDCG: 0.8269
Средний Precision: 0.5715
Средний Recall: 0.7817
Средний F1-Score: 0.6303
🏃 View run 19 at: http://127.0.0.1:5000/#/experiments/1/runs/47cc7dc7a4a6481e8c226f74c8ef0522
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Number of finished trials: 20
Best params: {'C': 0.19394744414409867, 'penalty': 'l2', 'solver': 'saga'}


In [48]:
study.best_trial.user_attrs['pipeline_params']

{'model__C': 0.19394744414409867,
 'model__penalty': 'l2',
 'model__solver': 'saga'}

In [49]:
pipeline_lr_best_optuna = Pipeline([
    ('preprocessing', ColumnTransformer([
        ('numeric_scaling', StandardScaler(), numeric_features),
        ('categorical_encoding', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ])),
    ('model', LogisticRegression(random_state=RANDOM_STATE))
])

pipeline_lr_best_optuna.set_params(**study.best_trial.user_attrs['pipeline_params'])
pipeline_lr_best_optuna.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric_scaling', ...), ('categorical_encoding', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of

In [50]:
y_pred_proba = pipeline_lr_best_optuna.predict_proba(X_test)

df_test = df.loc[X_test.index].copy()
df_test['y_pred_proba'] = y_pred_proba[:, 1]

In [51]:
ndcg_lr, precision_lr, recall_lr, f1_lr = calculate_metrics(df_test)

metrics_lr = {
    'NDCG': ndcg_lr,
    'Precision': precision_lr,
    'Recall': recall_lr,
    'F1': f1_lr
}

Средний NDCG: 0.7516
Средний Precision: 0.6371
Средний Recall: 0.5965
Средний F1-Score: 0.6003


In [52]:
RUN_NAME = "best_optuna_lr" 
REGISTRY_MODEL_NAME = "best_optuna_lr"

In [53]:
signature = mlflow.models.infer_signature(X_test, y_test)
input_example = X_test[:10]
code_paths = ["ML_Experiments.ipynb"]

try:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
except:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

    lr_info = mlflow.sklearn.log_model(sk_model=pipeline_lr_best_optuna, 
                                       artifact_path='best_optuna_lr',
                                       registered_model_name=REGISTRY_MODEL_NAME,
                                       input_example=input_example,
                                       code_paths=code_paths,
                                       await_registration_for=60
                                      )
    mlflow.log_metrics(metrics_lr)
    mlflow.log_params(best_params_optuna)

Successfully registered model 'best_optuna_lr'.
2026/05/08 16:24:38 INFO mlflow.store.model_registry.abstract_store: Waiting up to 60 seconds for model version to finish creation. Model name: best_optuna_lr, version 1


🏃 View run best_optuna_lr at: http://127.0.0.1:5000/#/experiments/1/runs/3ccd2c4f94c44a8f9da7432ac80ac2d1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Created version '1' of model 'best_optuna_lr'.


# RandomForestClassifier

In [54]:
def objective(trial: optuna.Trial) -> float:
    params = {
        'model__n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=50),
        'model__max_depth': trial.suggest_int('max_depth', 3, 30),
        'model__min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'model__min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20)
    }
    
    pipeline_rf = Pipeline([
        ('preprocessing', ColumnTransformer([
            ('numeric_scaling', StandardScaler(), numeric_features),
            ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
        ])),
        ('model', RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced'))
    ])
    
    pipeline_rf.set_params(**params)
    
    kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    
    for train_idx, val_idx in kfold.split(X_train, y_train):
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        pipeline_rf.fit(X_fold_train, y_fold_train)
        y_pred_proba = pipeline_rf.predict_proba(X_fold_val)
        

        df_val = df.loc[X_fold_val.index].copy()
        df_val['y_pred_proba'] = y_pred_proba[:, 1]
        
        ndcg, _, _, _ = calculate_metrics(df_val)

        trial.set_user_attr('pipeline_params', params)
    
    return ndcg

In [55]:
try:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
except:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)

RUN_NAME_OPTUNE = 'RandomForestClassifier_optuna'

with mlflow.start_run(run_name=RUN_NAME_OPTUNE, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

🏃 View run RandomForestClassifier_optuna at: http://127.0.0.1:5000/#/experiments/1/runs/b194102d3ba346fa850a1c0eba59f125
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [56]:
STUDY_NAME = "RandomForestClassifier_optuna"

mlflc = MLflowCallback(
    tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
    metric_name="NDCG",
    create_experiment=False,
    mlflow_kwargs={'experiment_id': experiment_id, 'tags': {MLFLOW_PARENT_RUN_ID: run_id}}
)

In [57]:
study = optuna.create_study(direction='maximize', 
                            sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                            study_name=STUDY_NAME,
                            storage=STUDY_DB_NAME,
                            load_if_exists=False)
study.optimize(objective, 
               n_trials=20, 
               callbacks=[mlflc]
              )
best_params_optuna = study.best_params

print(f"Number of finished trials: {len(study.trials)}")
print(f"Best params: {best_params_optuna}")

[I 2026-05-08 16:24:38,851] A new study created in RDB with name: RandomForestClassifier_optuna


Средний NDCG: 0.8564
Средний Precision: 0.6951
Средний Recall: 0.8292
Средний F1-Score: 0.7345
Средний NDCG: 0.8447
Средний Precision: 0.6848
Средний Recall: 0.8185
Средний F1-Score: 0.7240


[I 2026-05-08 16:26:22,825] Trial 0 finished with value: 0.8523321436552282 and parameters: {'n_estimators': 450, 'max_depth': 29, 'min_samples_split': 15, 'min_samples_leaf': 12}. Best is trial 0 with value: 0.8523321436552282.


Средний NDCG: 0.8523
Средний Precision: 0.6910
Средний Recall: 0.8239
Средний F1-Score: 0.7303
🏃 View run 0 at: http://127.0.0.1:5000/#/experiments/1/runs/15465de5aaca41de8841b8f8273c5ba0
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8332
Средний Precision: 0.6222
Средний Recall: 0.7997
Средний F1-Score: 0.6726
Средний NDCG: 0.8211
Средний Precision: 0.5975
Средний Recall: 0.7907
Средний F1-Score: 0.6518


[I 2026-05-08 16:26:57,471] Trial 1 finished with value: 0.8295085461850679 and parameters: {'n_estimators': 200, 'max_depth': 7, 'min_samples_split': 3, 'min_samples_leaf': 18}. Best is trial 0 with value: 0.8523321436552282.


...

Средний NDCG: 0.8546
Средний Precision: 0.7243
Средний Recall: 0.8152
Средний F1-Score: 0.7484
🏃 View run 17 at: http://127.0.0.1:5000/#/experiments/1/runs/ce4c40a3f87f4918ae1c254c45c0d256
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8595
Средний Precision: 0.7456
Средний Recall: 0.8148
Средний F1-Score: 0.7614
Средний NDCG: 0.8477
Средний Precision: 0.7323
Средний Recall: 0.8015
Средний F1-Score: 0.7483


[I 2026-05-08 16:59:31,124] Trial 18 finished with value: 0.855576057230516 and parameters: {'n_estimators': 750, 'max_depth': 26, 'min_samples_split': 17, 'min_samples_leaf': 1}. Best is trial 16 with value: 0.8557538330964121.


Средний NDCG: 0.8556
Средний Precision: 0.7382
Средний Recall: 0.8071
Средний F1-Score: 0.7541
🏃 View run 18 at: http://127.0.0.1:5000/#/experiments/1/runs/5f8c35e89d2c4ae39e77026837e0a187
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8555
Средний Precision: 0.6860
Средний Recall: 0.8306
Средний F1-Score: 0.7292
Средний NDCG: 0.8441
Средний Precision: 0.6733
Средний Recall: 0.8199
Средний F1-Score: 0.7170


[I 2026-05-08 17:13:30,591] Trial 19 finished with value: 0.8514907068762502 and parameters: {'n_estimators': 1000, 'max_depth': 30, 'min_samples_split': 18, 'min_samples_leaf': 15}. Best is trial 16 with value: 0.8557538330964121.


Средний NDCG: 0.8515
Средний Precision: 0.6815
Средний Recall: 0.8254
Средний F1-Score: 0.7248
🏃 View run 19 at: http://127.0.0.1:5000/#/experiments/1/runs/41839114667745bdb509f437d14b415b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Number of finished trials: 20
Best params: {'n_estimators': 800, 'max_depth': 26, 'min_samples_split': 17, 'min_samples_leaf': 1}


In [58]:
study.best_trial.user_attrs['pipeline_params']

{'model__n_estimators': 800,
 'model__max_depth': 26,
 'model__min_samples_split': 17,
 'model__min_samples_leaf': 1}

In [59]:
pipeline_rf_best_optuna = Pipeline([
        ('preprocessing', ColumnTransformer([
            ('numeric_scaling', StandardScaler(), numeric_features),
            ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
        ])),
        ('model', RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced'))
    ])

pipeline_rf_best_optuna.set_params(**study.best_trial.user_attrs['pipeline_params'])
pipeline_rf_best_optuna.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric_scaling', ...), ('categorical_encoding', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of

In [60]:
y_pred_proba = pipeline_rf_best_optuna.predict_proba(X_test)

df_test = df.loc[X_test.index].copy()
df_test['y_pred_proba'] = y_pred_proba[:, 1]

In [61]:
ndcg_rf, precision_rf, recall_rf, f1_rf = calculate_metrics(df_test)

metrics_rf = {
    'NDCG': ndcg_rf,
    'Precision': precision_rf,
    'Recall': recall_rf,
    'F1': f1_rf
}

Средний NDCG: 0.7776
Средний Precision: 0.6830
Средний Recall: 0.7355
Средний F1-Score: 0.6934


In [62]:
RUN_NAME = "best_optuna_rf" 
REGISTRY_MODEL_NAME = "best_optuna_rf"

In [63]:
signature = mlflow.models.infer_signature(X_test, y_test)
input_example = X_test[:10]
code_paths = ["ML_Experiments.ipynb"]

try:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
except:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

    lr_info = mlflow.sklearn.log_model(sk_model=pipeline_rf_best_optuna, 
                                       artifact_path='best_optuna_rf',
                                       registered_model_name=REGISTRY_MODEL_NAME,
                                       input_example=input_example,
                                       code_paths=code_paths,
                                       await_registration_for=60
                                      )
    mlflow.log_metrics(metrics_rf)
    mlflow.log_params(best_params_optuna)

Successfully registered model 'best_optuna_rf'.
2026/05/08 17:15:02 INFO mlflow.store.model_registry.abstract_store: Waiting up to 60 seconds for model version to finish creation. Model name: best_optuna_rf, version 1


🏃 View run best_optuna_rf at: http://127.0.0.1:5000/#/experiments/1/runs/886fbc90fa7a45ada1c14aa7a5ed25e3
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Created version '1' of model 'best_optuna_rf'.


# XGBClassifier

In [64]:
def objective_xgb(trial: optuna.Trial) -> float:
    params = {
        'model__n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=50),
        'model__max_depth': trial.suggest_int('max_depth', 3, 15),
        'model__learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'model__min_child_weight': trial.suggest_int('min_child_weight', 1, 10)
    }
    
    pipeline_xgb = Pipeline([
        ('preprocessing', ColumnTransformer([
            ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
        ], remainder='passthrough')),
        ('model', XGBClassifier(
            random_state=RANDOM_STATE, 
            eval_metric='logloss', 
            use_label_encoder=False, 
            verbosity=0,
            scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train)
        ))
    ])
    
    pipeline_xgb.set_params(**params)
    
    kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    
    for train_idx, val_idx in kfold.split(X_train, y_train):
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        pipeline_xgb.fit(X_fold_train, y_fold_train)
        y_pred_proba = pipeline_xgb.predict_proba(X_fold_val)
        
        df_val = df.loc[X_fold_val.index].copy()
        df_val['y_pred_proba'] = y_pred_proba[:, 1]
        
        ndcg, _, _, _ = calculate_metrics(df_val)
        
        trial.set_user_attr('pipeline_params', params)
    
    return ndcg

In [65]:
try:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
except:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)

RUN_NAME_OPTUNE_XGB = 'XGBClassifier_optuna'

with mlflow.start_run(run_name=RUN_NAME_OPTUNE_XGB, experiment_id=experiment_id) as run:
    run_id_xgb = run.info.run_id

🏃 View run XGBClassifier_optuna at: http://127.0.0.1:5000/#/experiments/1/runs/ee3f6916f8ec4dc3b614739e8bca15f5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [66]:
STUDY_NAME_XGB = "XGBClassifier_optuna"

mlflc_xgb = MLflowCallback(
    tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
    metric_name="NDCG",
    create_experiment=False,
    mlflow_kwargs={'experiment_id': experiment_id, 'tags': {MLFLOW_PARENT_RUN_ID: run_id_xgb}}
)

In [67]:
study_xgb = optuna.create_study(direction='maximize', 
                                sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                                study_name=STUDY_NAME_XGB,
                                storage=STUDY_DB_NAME,
                                load_if_exists=False)

study_xgb.optimize(objective_xgb, 
                   n_trials=20, 
                   callbacks=[mlflc_xgb]
                  )

best_params_xgb = study_xgb.best_params
print(f"Number of finished trials: {len(study_xgb.trials)}")
print(f"Best params XGBoost: {best_params_xgb}")

[I 2026-05-08 17:15:02,200] A new study created in RDB with name: XGBClassifier_optuna


Средний NDCG: 0.8600
Средний Precision: 0.7702
Средний Recall: 0.8029
Средний F1-Score: 0.7708
Средний NDCG: 0.8474
Средний Precision: 0.7509
Средний Recall: 0.7874
Средний F1-Score: 0.7534


[I 2026-05-08 17:15:22,727] Trial 0 finished with value: 0.8558738164850641 and parameters: {'n_estimators': 450, 'max_depth': 15, 'learning_rate': 0.1205712628744377, 'min_child_weight': 6}. Best is trial 0 with value: 0.8558738164850641.


Средний NDCG: 0.8559
Средний Precision: 0.7618
Средний Recall: 0.7934
Средний F1-Score: 0.7620
🏃 View run 0 at: http://127.0.0.1:5000/#/experiments/1/runs/ce9c9e5208da4a809793aa36be09a522
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8386
Средний Precision: 0.6220
Средний Recall: 0.8042
Средний F1-Score: 0.6742
Средний NDCG: 0.8291
Средний Precision: 0.6081
Средний Recall: 0.7930
Средний F1-Score: 0.6597


[I 2026-05-08 17:15:36,277] Trial 1 finished with value: 0.8353950197667808 and parameters: {'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.012184186502221764, 'min_child_weight': 9}. Best is trial 0 with value: 0.8558738164850641.


...

Средний NDCG: 0.8568
Средний Precision: 0.7616
Средний Recall: 0.7995
Средний F1-Score: 0.7647
🏃 View run 17 at: http://127.0.0.1:5000/#/experiments/1/runs/11fd108b49bf421382aeab8870cb481b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8550
Средний Precision: 0.6724
Средний Recall: 0.8353
Средний F1-Score: 0.7221
Средний NDCG: 0.8446
Средний Precision: 0.6637
Средний Recall: 0.8200
Средний F1-Score: 0.7096


[I 2026-05-08 17:20:17,177] Trial 18 finished with value: 0.8517045537157859 and parameters: {'n_estimators': 250, 'max_depth': 6, 'learning_rate': 0.0470590358414764, 'min_child_weight': 4}. Best is trial 14 with value: 0.8574828172181674.


Средний NDCG: 0.8517
Средний Precision: 0.6711
Средний Recall: 0.8284
Средний F1-Score: 0.7187
🏃 View run 18 at: http://127.0.0.1:5000/#/experiments/1/runs/759d5ba4b7d043d6abaa4432b5dc3bfc
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8601
Средний Precision: 0.7633
Средний Recall: 0.8121
Средний F1-Score: 0.7710
Средний NDCG: 0.8478
Средний Precision: 0.7506
Средний Recall: 0.8027
Средний F1-Score: 0.7606


[I 2026-05-08 17:20:33,530] Trial 19 finished with value: 0.8573114855204316 and parameters: {'n_estimators': 300, 'max_depth': 10, 'learning_rate': 0.18266145263423655, 'min_child_weight': 7}. Best is trial 14 with value: 0.8574828172181674.


Средний NDCG: 0.8573
Средний Precision: 0.7569
Средний Recall: 0.8053
Средний F1-Score: 0.7652
🏃 View run 19 at: http://127.0.0.1:5000/#/experiments/1/runs/5b9ddd91dc5a4e719757279a3f438bdd
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Number of finished trials: 20
Best params XGBoost: {'n_estimators': 250, 'max_depth': 7, 'learning_rate': 0.16401332399717264, 'min_child_weight': 7}


In [68]:
pipeline_xgb_best = Pipeline([
    ('preprocessing', ColumnTransformer([
        ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
    ], remainder='passthrough')),
    ('model', XGBClassifier(
        random_state=RANDOM_STATE, 
        eval_metric='logloss', 
        use_label_encoder=False, 
        verbosity=0,
        scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train)
    ))
])

pipeline_xgb_best.set_params(**study_xgb.best_trial.user_attrs['pipeline_params'])
pipeline_xgb_best.fit(X_train, y_train)

y_pred_proba_xgb = pipeline_xgb_best.predict_proba(X_test)

df_test_xgb = df.loc[X_test.index].copy()
df_test_xgb['y_pred_proba'] = y_pred_proba_xgb[:, 1]

ndcg_xgb, precision_xgb, recall_xgb, f1_xgb = calculate_metrics(df_test_xgb)
metrics_xgb = {
    'NDCG': ndcg_xgb,
    'Precision': precision_xgb,
    'Recall': recall_xgb,
    'F1': f1_xgb
}

Средний NDCG: 0.7790
Средний Precision: 0.6673
Средний Recall: 0.7522
Средний F1-Score: 0.6915


In [69]:
RUN_NAME_XGB = "best_optuna_xgb"
REGISTRY_MODEL_NAME_XGB = "best_optuna_xgb"

signature = mlflow.models.infer_signature(X_test, y_test)
input_example = X_test[:10]
code_paths = ["ML_Experiments.ipynb"]

with mlflow.start_run(run_name=RUN_NAME_XGB, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    
    xgb_info = mlflow.sklearn.log_model(sk_model=pipeline_xgb_best, 
                                       artifact_path='best_optuna_xgb',
                                       registered_model_name=REGISTRY_MODEL_NAME_XGB,
                                       input_example=input_example,
                                       code_paths=code_paths,
                                       await_registration_for=60
                                      )
    mlflow.log_metrics(metrics_xgb)
    mlflow.log_params(best_params_xgb)

Successfully registered model 'best_optuna_xgb'.
2026/05/08 17:20:40 INFO mlflow.store.model_registry.abstract_store: Waiting up to 60 seconds for model version to finish creation. Model name: best_optuna_xgb, version 1


🏃 View run best_optuna_xgb at: http://127.0.0.1:5000/#/experiments/1/runs/5fd911c2983c4196a762137b4a0f0444
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Created version '1' of model 'best_optuna_xgb'.


# LGBMClassifier

In [70]:
def objective_lgbm(trial: optuna.Trial) -> float:
    params = {
        'model__n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=50),
        'model__max_depth': trial.suggest_int('max_depth', 3, 15),
        'model__learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'model__num_leaves': trial.suggest_int('num_leaves', 20, 300),
        'model__min_child_samples': trial.suggest_int('min_child_samples', 5, 100)
    }
    
    pipeline_lgbm = Pipeline([
        ('preprocessing', ColumnTransformer([
            ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
        ], remainder='passthrough')),
        ('model', LGBMClassifier(
            random_state=RANDOM_STATE, 
            verbose=-1,
            class_weight='balanced'
        ))
    ])
    
    pipeline_lgbm.set_params(**params)
    
    kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    
    for train_idx, val_idx in kfold.split(X_train, y_train):
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        pipeline_lgbm.fit(X_fold_train, y_fold_train)
        y_pred_proba = pipeline_lgbm.predict_proba(X_fold_val)
        
        df_val = df.loc[X_fold_val.index].copy()
        df_val['y_pred_proba'] = y_pred_proba[:, 1]
        
        ndcg, _, _, _ = calculate_metrics(df_val)
        
        trial.set_user_attr('pipeline_params', params)
    
    return ndcg

In [71]:
RUN_NAME_OPTUNE_LGBM = 'LGBMClassifier_optuna'

with mlflow.start_run(run_name=RUN_NAME_OPTUNE_LGBM, experiment_id=experiment_id) as run:
    run_id_lgbm = run.info.run_id

🏃 View run LGBMClassifier_optuna at: http://127.0.0.1:5000/#/experiments/1/runs/a19ad18001e44077bcce90509eeaa80a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [72]:
STUDY_NAME_LGBM = "LGBMClassifier_optuna"

mlflc_lgbm = MLflowCallback(
    tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
    metric_name="NDCG",
    create_experiment=False,
    mlflow_kwargs={'experiment_id': experiment_id, 'tags': {MLFLOW_PARENT_RUN_ID: run_id_lgbm}}
)

In [ ]:
study_lgbm = optuna.create_study(direction='maximize', 
                                 sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                                 study_name=STUDY_NAME_LGBM,
                                 storage=STUDY_DB_NAME,
                                 load_if_exists=False)

study_lgbm.optimize(objective_lgbm, 
                    n_trials=20, 
                    callbacks=[mlflc_lgbm]
                   )

best_params_lgbm = study_lgbm.best_params
print(f"Number of finished trials: {len(study_lgbm.trials)}")
print(f"Best params LGBM: {best_params_lgbm}")

[I 2026-05-08 17:20:40,681] A new study created in RDB with name: LGBMClassifier_optuna


Средний NDCG: 0.8603
Средний Precision: 0.7741
Средний Recall: 0.7994
Средний F1-Score: 0.7718
Средний NDCG: 0.8472
Средний Precision: 0.7656
Средний Recall: 0.7870
Средний F1-Score: 0.7619


[I 2026-05-08 17:21:57,115] Trial 0 finished with value: 0.85649136123959 and parameters: {'n_estimators': 450, 'max_depth': 15, 'learning_rate': 0.1205712628744377, 'num_leaves': 188, 'min_child_samples': 19}. Best is trial 0 with value: 0.85649136123959.


Средний NDCG: 0.8565
Средний Precision: 0.7675
Средний Recall: 0.7869
Средний F1-Score: 0.7622
🏃 View run 0 at: http://127.0.0.1:5000/#/experiments/1/runs/1b42fdbc9fdc41ce98ff6a3794848627
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8525
Средний Precision: 0.6547
Средний Recall: 0.8323
Средний F1-Score: 0.7091
Средний NDCG: 0.8432
Средний Precision: 0.6440
Средний Recall: 0.8218
Средний F1-Score: 0.6971


[I 2026-05-08 17:22:10,963] Trial 1 finished with value: 0.8492437739747656 and parameters: {'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.19030368381735815, 'num_leaves': 188, 'min_child_samples': 72}. Best is trial 0 with value: 0.85649136123959.


...

Средний NDCG: 0.8538
Средний Precision: 0.6756
Средний Recall: 0.8332
Средний F1-Score: 0.7240
🏃 View run 16 at: http://127.0.0.1:5000/#/experiments/1/runs/1c07043733e94de19c5ea9576f61b174
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8611
Средний Precision: 0.7585
Средний Recall: 0.8199
Средний F1-Score: 0.7724
Средний NDCG: 0.8487
Средний Precision: 0.7493
Средний Recall: 0.8086
Средний F1-Score: 0.7623


[I 2026-05-08 17:33:50,299] Trial 17 finished with value: 0.8574469512820642 and parameters: {'n_estimators': 550, 'max_depth': 13, 'learning_rate': 0.1357039061098737, 'num_leaves': 59, 'min_child_samples': 80}. Best is trial 12 with value: 0.8578275187630338.


Средний NDCG: 0.8574
Средний Precision: 0.7528
Средний Recall: 0.8113
Средний F1-Score: 0.7652
🏃 View run 17 at: http://127.0.0.1:5000/#/experiments/1/runs/ef5b2222387b453b87765281ce5126c2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8606
Средний Precision: 0.7608
Средний Recall: 0.8150
Средний F1-Score: 0.7716
Средний NDCG: 0.8480
Средний Precision: 0.7521
Средний Recall: 0.8033
Средний F1-Score: 0.7623


[I 2026-05-08 17:34:27,829] Trial 18 finished with value: 0.8567673720184357 and parameters: {'n_estimators': 500, 'max_depth': 13, 'learning_rate': 0.16696881791330048, 'num_leaves': 65, 'min_child_samples': 90}. Best is trial 12 with value: 0.8578275187630338.


Средний NDCG: 0.8568
Средний Precision: 0.7569
Средний Recall: 0.8066
Средний F1-Score: 0.7664
🏃 View run 18 at: http://127.0.0.1:5000/#/experiments/1/runs/3f7b8b92289a4f698b035708584f3a68
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [ ]:
pipeline_lgbm_best = Pipeline([
    ('preprocessing', ColumnTransformer([
        ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
    ], remainder='passthrough')),
    ('model', LGBMClassifier(
        random_state=RANDOM_STATE, 
        verbose=-1,
        class_weight='balanced'
    ))
])

pipeline_lgbm_best.set_params(**study_lgbm.best_trial.user_attrs['pipeline_params'])
pipeline_lgbm_best.fit(X_train, y_train)

y_pred_proba_lgbm = pipeline_lgbm_best.predict_proba(X_test)

df_test_lgbm = df.loc[X_test.index].copy()
df_test_lgbm['y_pred_proba'] = y_pred_proba_lgbm[:, 1]

ndcg_lgbm, precision_lgbm, recall_lgbm, f1_lgbm = calculate_metrics(df_test_lgbm)
metrics_lgbm = {
    'NDCG': ndcg_lgbm,
    'Precision': precision_lgbm,
    'Recall': recall_lgbm,
    'F1': f1_lgbm
}

In [ ]:
RUN_NAME_LGBM = "best_optuna_lgbm"
REGISTRY_MODEL_NAME_LGBM = "best_optuna_lgbm"

signature = mlflow.models.infer_signature(X_test, y_test)
input_example = X_test[:10]
code_paths = ["ML_Experiments.ipynb"]

with mlflow.start_run(run_name=RUN_NAME_LGBM, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    
    lgbm_info = mlflow.sklearn.log_model(sk_model=pipeline_lgbm_best, 
                                        artifact_path='best_optuna_lgbm',
                                        registered_model_name=REGISTRY_MODEL_NAME_LGBM,
                                        input_example=input_example,
                                        code_paths=code_paths,
                                        await_registration_for=60
                                       )
    mlflow.log_metrics(metrics_lgbm)
    mlflow.log_params(best_params_lgbm)

# CatBoostClassifier

In [ ]:
def objective_catboost(trial: optuna.Trial) -> float:
    params = {
        'model__iterations': trial.suggest_int('iterations', 100, 1000, step=50),
        'model__depth': trial.suggest_int('depth', 3, 10),
        'model__learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'model__l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10, log=True)
    }
    
    pipeline_catboost = Pipeline([
        ('preprocessing', ColumnTransformer([
            ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
        ], remainder='passthrough')),
        ('model', CatBoostClassifier(
            random_state=RANDOM_STATE, 
            verbose=0, 
            auto_class_weights='Balanced'
        ))
    ])
    
    pipeline_catboost.set_params(**params)
    
    kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    
    for train_idx, val_idx in kfold.split(X_train, y_train):
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        pipeline_catboost.fit(X_fold_train, y_fold_train)
        y_pred_proba = pipeline_catboost.predict_proba(X_fold_val)
        
        df_val = df.loc[X_fold_val.index].copy()
        df_val['y_pred_proba'] = y_pred_proba[:, 1]
        
        ndcg, _, _, _ = calculate_metrics(df_val)
        
        trial.set_user_attr('pipeline_params', params)
    
    return ndcg

In [ ]:
RUN_NAME_OPTUNE_CATBOOST = 'CatBoostClassifier_optuna'

with mlflow.start_run(run_name=RUN_NAME_OPTUNE_CATBOOST, experiment_id=experiment_id) as run:
    run_id_catboost = run.info.run_id

In [ ]:
STUDY_NAME_CATBOOST = "CatBoostClassifier_optuna"

mlflc_catboost = MLflowCallback(
    tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
    metric_name="NDCG",
    create_experiment=False,
    mlflow_kwargs={'experiment_id': experiment_id, 'tags': {MLFLOW_PARENT_RUN_ID: run_id_catboost}}
)

In [ ]:
study_catboost = optuna.create_study(direction='maximize', 
                                     sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                                     study_name=STUDY_NAME_CATBOOST,
                                     storage=STUDY_DB_NAME,
                                     load_if_exists=False)

study_catboost.optimize(objective_catboost, 
                        n_trials=20, 
                        callbacks=[mlflc_catboost]
                       )

best_params_catboost = study_catboost.best_params
print(f"Number of finished trials: {len(study_catboost.trials)}")
print(f"Best params CatBoost: {best_params_catboost}")

In [ ]:
pipeline_catboost_best = Pipeline([
    ('preprocessing', ColumnTransformer([
        ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
    ], remainder='passthrough')),
    ('model', CatBoostClassifier(
        random_state=RANDOM_STATE, 
        verbose=0, 
        auto_class_weights='Balanced'
    ))
])

pipeline_catboost_best.set_params(**study_catboost.best_trial.user_attrs['pipeline_params'])
pipeline_catboost_best.fit(X_train, y_train)

y_pred_proba_catboost = pipeline_catboost_best.predict_proba(X_test)

df_test_catboost = df.loc[X_test.index].copy()
df_test_catboost['y_pred_proba'] = y_pred_proba_catboost[:, 1]

ndcg_catboost, precision_catboost, recall_catboost, f1_catboost = calculate_metrics(df_test_catboost)
metrics_catboost = {
    'NDCG': ndcg_catboost,
    'Precision': precision_catboost,
    'Recall': recall_catboost,
    'F1': f1_catboost
}

In [ ]:
RUN_NAME_CATBOOST = "best_optuna_catboost"
REGISTRY_MODEL_NAME_CATBOOST = "best_optuna_catboost"

signature = mlflow.models.infer_signature(X_test, y_test)
input_example = X_test[:10]
code_paths = ["ML_Experiments.ipynb"]

with mlflow.start_run(run_name=RUN_NAME_CATBOOST, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    
    catboost_info = mlflow.sklearn.log_model(sk_model=pipeline_catboost_best, 
                                             artifact_path='best_optuna_catboost',
                                             registered_model_name=REGISTRY_MODEL_NAME_CATBOOST,
                                             input_example=input_example,
                                             code_paths=code_paths,
                                             await_registration_for=60
                                            )
    mlflow.log_metrics(metrics_catboost)
    mlflow.log_params(best_params_catboost)

# Model comparison

In [ ]:
models_comparison = pd.DataFrame({
    'Model': ['LogisticRegression', 'RandomForest', 'XGBoost', 'LightGBM', 'CatBoost'],
    'NDCG': [ndcg_lr, ndcg_rf, 
             ndcg_xgb, ndcg_lgbm, ndcg_catboost],
    'Precision': [precision_lr, precision_rf, 
                  precision_xgb, precision_lgbm, precision_catboost],
    'Recall': [recall_lr, recall_rf, 
               recall_xgb, recall_lgbm, recall_catboost],
    'F1': [f1_lr, f1_rf, 
           f1_xgb, f1_lgbm, f1_catboost]
})

models_comparison

In [ ]:
best_model_idx = models_comparison['NDCG'].idxmax()
print(f"\nЛучшая модель: {models_comparison.iloc[best_model_idx]['Model']} с NDCG = {models_comparison.iloc[best_model_idx]['NDCG']:.4f}")

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

По итогам подбора гиперпараметров лучшее качество показала модель `XGBClassifier`. Теперь попробуем добавить дополнительные признаки от кандидата генераторов. В первую очередь используем коллаборативную фильтрацию.

</div>

# ALS

## Feature engineering

In [ ]:
def create_interaction_matrix(df):
    unique_vacancies = df['vacancy_id'].unique().tolist()
    unique_resumes = df['resume_id'].unique().tolist()

    id2vacancy = dict(enumerate(unique_vacancies))
    id2resume = dict(enumerate(unique_resumes))
    vacancy2id = {v: k for k, v in id2vacancy.items()}
    resume2id = {v: k for k, v in id2resume.items()}

    rows = [vacancy2id[vacancy] for vacancy in df['vacancy_id']]
    cols = [resume2id[resume] for resume in df['resume_id']]

    interactions_matrix = csr_matrix(
        (df['target'], (rows, cols)),
        shape=(len(unique_vacancies), len(unique_resumes)),
        dtype=np.float32,
    )

    return interactions_matrix, id2vacancy, id2resume, vacancy2id, resume2id, unique_vacancies, unique_resumes

In [ ]:
interactions_matrix, id2vacancy, id2resume, vacancy2id, resume2id, unique_vacancies, unique_resumes = create_interaction_matrix(df)

print(f"Размерность матрицы взаимодействий: {interactions_matrix.shape}")
print(f"Плотность матрицы: {interactions_matrix.nnz / (interactions_matrix.shape[0] * interactions_matrix.shape[1]):.6f}")

In [ ]:
def get_factors(interactions_matrix):
    als_model = AlternatingLeastSquares(
        factors=64,          
        regularization=0.1,  
        iterations=30,       
        random_state=RANDOM_STATE,
        num_threads=0
    )
    
    als_model.fit(interactions_matrix.T)

    vacancy_factors = als_model.item_factors
    resume_factors = als_model.user_factors

    return vacancy_factors, resume_factors

In [ ]:
vacancy_factors, resume_factors = get_factors(interactions_matrix)

print(f"Размерность эмбеддингов вакансий: {vacancy_factors.shape}")
print(f"Размерность эмбеддингов резюме: {resume_factors.shape}")

In [ ]:
def get_als_score(vacancy_id, resume_id):
    if vacancy_id not in vacancy2id or resume_id not in resume2id:
        return 0
    else:
        vac_idx = vacancy2id[vacancy_id]
        res_idx = resume2id[resume_id]
        score = np.dot(vacancy_factors[vac_idx], resume_factors[res_idx])
        return score

df['als_score'] = df.apply(
    lambda row: get_als_score(row['vacancy_id'], row['resume_id']), 
    axis=1
)

In [ ]:
df[['vacancy_id', 'resume_id', 'target', 'als_score']].head()

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Проверим схожесть резюме по латентным векторам.

</div>

In [ ]:
def find_similar_resumes(resume_id, resume_factors, n_similar=10, metric='cosine'):
    """
    Находит N наиболее похожих резюме на заданное.
    """
    if resume_id not in resume2id:
        print(f"Резюме с ID {resume_id} не найдено")
        return []

    target_idx = resume2id[resume_id]
    target_vector = resume_factors[target_idx].reshape(1, -1)
    
    if metric == 'cosine':
        similarities = cosine_similarity(target_vector, resume_factors)[0]
    else:
        similarities = np.dot(resume_factors, target_vector.T).flatten()
    
    similar_indices = np.argsort(similarities)[::-1][1:n_similar+1]
    
    similar_resumes = []
    for idx in similar_indices:
        sim_resume_id = unique_resumes[idx]
        similarity_score = similarities[idx]
        similar_resumes.append((sim_resume_id, similarity_score))
    
    return similar_resumes

def get_resume_profile(resume_id, df):
    """Вспомогательная функция для получения информации о резюме"""
    resume_data = df[df['resume_id'] == resume_id].iloc[0]
    return {
        'resume_id': resume_id,
        'title': resume_data.get('resume_title', 'N/A'),
        'specialization': resume_data.get('resume_specialization', 'N/A'),
        'skills': resume_data.get('resume_skills', 'N/A'),
        'experience': resume_data.get('resume_total_experience', 'N/A'),
        'salary': resume_data.get('resume_salary', 'N/A'),
        'location': resume_data.get('resume_location', 'N/A')
    }

In [ ]:
example_resume_id = df['resume_id'].sample(1, random_state=RANDOM_STATE).values[0].item()
print(f"Исходное резюме (ID: {example_resume_id}):")
original_profile = get_resume_profile(example_resume_id, df)
for key, value in original_profile.items():
    print(f"  {key}: {value}")

print(f"\nТоп-5 похожих резюме (косинусная близость):")
similar_resumes = find_similar_resumes(example_resume_id, resume_factors, n_similar=5, metric='cosine')

for i, (sim_id, score) in enumerate(similar_resumes, 1):
    profile = get_resume_profile(sim_id, df)
    print(f"\n{i}. Резюме ID: {sim_id} (сходство: {score:.4f})")
    print(f"   Должность: {profile['title']}")
    print(f"   Специализация: {profile['specialization']}")
    print(f"   Локация: {profile['location']}")
    print(f"   Опыт: {profile['experience']}")

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Разделим тренировочную выборку еще на две, чтобы избежать подглядывания и добавить в признак `als_score` нулевые значения в тренировочный датасет, т.к. у нас могут быть вакансии и резюме без взаимодействий до обучения.

</div>

In [ ]:
X_train_als, X_test_als, y_train_als, y_test_als = train_test_split(X_train, y_train, test_size=0.3, random_state=RANDOM_STATE, stratify=y_train)

In [ ]:
train_als_interactions = df.loc[X_train_als.index, ['vacancy_id', 'resume_id', 'target']]

interactions_matrix, id2vacancy, id2resume, vacancy2id, resume2id, unique_vacancies, unique_resumes = create_interaction_matrix(train_als_interactions)
vacancy_factors, resume_factors = get_factors(interactions_matrix)

# возвращаем айдишники вакансий и резюме для расчета als score
X_train[['vacancy_id', 'resume_id']] = df.loc[X_train.index, ['vacancy_id', 'resume_id']]
X_train['als_score'] = X_train.apply(
    lambda row: get_als_score(row['vacancy_id'], row['resume_id']), 
    axis=1
)

X_train = X_train.drop(['vacancy_id', 'resume_id'], axis=1)

In [ ]:
# проверим есть ли вакансии и резюме без взаимодействий до обучения
X_train[X_train['als_score'] == 0]

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Теперь добавим признак `als_score` для тестовой выборке, но уже возьмем матрицу интеракций на полной тренировочной выборке.

</div>

In [ ]:
train_interactions = df.loc[X_train.index, ['vacancy_id', 'resume_id', 'target']]

interactions_matrix, id2vacancy, id2resume, vacancy2id, resume2id, unique_vacancies, unique_resumes = create_interaction_matrix(train_interactions)
vacancy_factors, resume_factors = get_factors(interactions_matrix)

# возвращаем айдишники вакансий и резюме для расчета als score
X_test[['vacancy_id', 'resume_id']] = df.loc[X_test.index, ['vacancy_id', 'resume_id']]
X_test['als_score'] = X_test.apply(
    lambda row: get_als_score(row['vacancy_id'], row['resume_id']), 
    axis=1
)

X_test = X_test.drop(['vacancy_id', 'resume_id'], axis=1)

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Выборки получены, обучим `XGBClassifier` с подбором гиперпараметров.

</div>

In [ ]:
X_train.info()

## XGBClassifier

In [ ]:
def objective_xgb(trial: optuna.Trial) -> float:
    params = {
        'model__n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=50),
        'model__max_depth': trial.suggest_int('max_depth', 3, 15),
        'model__learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'model__min_child_weight': trial.suggest_int('min_child_weight', 1, 10)
    }
    
    pipeline_xgb = Pipeline([
        ('preprocessing', ColumnTransformer([
            ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
        ], remainder='passthrough')),
        ('model', XGBClassifier(
            random_state=RANDOM_STATE, 
            eval_metric='logloss', 
            use_label_encoder=False, 
            verbosity=0,
            scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train)
        ))
    ])
    
    pipeline_xgb.set_params(**params)
    
    kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    
    for train_idx, val_idx in kfold.split(X_train, y_train):
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        pipeline_xgb.fit(X_fold_train, y_fold_train)
        y_pred_proba = pipeline_xgb.predict_proba(X_fold_val)
        
        df_val = df.loc[X_fold_val.index].copy()
        df_val['y_pred_proba'] = y_pred_proba[:, 1]
        
        ndcg, _, _, _ = calculate_metrics(df_val)
        
        trial.set_user_attr('pipeline_params', params)
    
    return ndcg

In [ ]:
try:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
except:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)

RUN_NAME_OPTUNE_XGB = 'XGBClassifier_optuna_als'

with mlflow.start_run(run_name=RUN_NAME_OPTUNE_XGB, experiment_id=experiment_id) as run:
    run_id_xgb = run.info.run_id

In [ ]:
STUDY_NAME_XGB = "XGBClassifier_optuna_als"

mlflc_xgb = MLflowCallback(
    tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
    metric_name="NDCG",
    create_experiment=False,
    mlflow_kwargs={'experiment_id': experiment_id, 'tags': {MLFLOW_PARENT_RUN_ID: run_id_xgb}}
)

In [ ]:
study_xgb = optuna.create_study(direction='maximize', 
                                sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                                study_name=STUDY_NAME_XGB,
                                storage=STUDY_DB_NAME,
                                load_if_exists=False)

study_xgb.optimize(objective_xgb, 
                   n_trials=20, 
                   callbacks=[mlflc_xgb]
                  )

best_params_xgb = study_xgb.best_params
print(f"Number of finished trials: {len(study_xgb.trials)}")
print(f"Best params XGBoost: {best_params_xgb}")

In [ ]:
pipeline_xgb_best = Pipeline([
    ('preprocessing', ColumnTransformer([
        ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
    ], remainder='passthrough')),
    ('model', XGBClassifier(
        random_state=RANDOM_STATE, 
        eval_metric='logloss', 
        use_label_encoder=False, 
        verbosity=0,
        scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train)
    ))
])

pipeline_xgb_best.set_params(**study_xgb.best_trial.user_attrs['pipeline_params'])
pipeline_xgb_best.fit(X_train, y_train)

y_pred_proba_xgb = pipeline_xgb_best.predict_proba(X_test)

df_test_xgb = df.loc[X_test.index].copy()
df_test_xgb['y_pred_proba'] = y_pred_proba_xgb[:, 1]

ndcg_xgb_als, precision_xgb_als, recall_xgb_als, f1_xgb_als = calculate_metrics(df_test_xgb)
metrics_xgb_als = {
    'NDCG': ndcg_xgb_als,
    'Precision': precision_xgb_als,
    'Recall': recall_xgb_als,
    'F1': f1_xgb_als
}

In [ ]:
RUN_NAME_XGB = "best_optuna_xgb_als"
REGISTRY_MODEL_NAME_XGB = "best_optuna_xgb_als"

signature = mlflow.models.infer_signature(X_test, y_test)
input_example = X_test[:10]
code_paths = ["ML_Experiments.ipynb"]

with mlflow.start_run(run_name=RUN_NAME_XGB, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    
    xgb_info = mlflow.sklearn.log_model(sk_model=pipeline_xgb_best, 
                                       artifact_path='best_optuna_xgb_als',
                                       registered_model_name=REGISTRY_MODEL_NAME_XGB,
                                       input_example=input_example,
                                       code_paths=code_paths,
                                       await_registration_for=60
                                      )
    mlflow.log_metrics(metrics_xgb_als)
    mlflow.log_params(best_params_xgb)

## LGBMClassifier

In [ ]:
def objective_lgbm(trial: optuna.Trial) -> float:
    params = {
        'model__n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=50),
        'model__max_depth': trial.suggest_int('max_depth', 3, 15),
        'model__learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'model__num_leaves': trial.suggest_int('num_leaves', 20, 300),
        'model__min_child_samples': trial.suggest_int('min_child_samples', 5, 100)
    }
    
    pipeline_lgbm = Pipeline([
        ('preprocessing', ColumnTransformer([
            ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
        ], remainder='passthrough')),
        ('model', LGBMClassifier(
            random_state=RANDOM_STATE, 
            verbose=-1,
            class_weight='balanced'
        ))
    ])
    
    pipeline_lgbm.set_params(**params)
    
    kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    
    for train_idx, val_idx in kfold.split(X_train, y_train):
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        pipeline_lgbm.fit(X_fold_train, y_fold_train)
        y_pred_proba = pipeline_lgbm.predict_proba(X_fold_val)
        
        df_val = df.loc[X_fold_val.index].copy()
        df_val['y_pred_proba'] = y_pred_proba[:, 1]
        
        ndcg, _, _, _ = calculate_metrics(df_val)
        
        trial.set_user_attr('pipeline_params', params)
    
    return ndcg

In [ ]:
RUN_NAME_OPTUNE_LGBM = 'LGBMClassifier_optuna_als'

with mlflow.start_run(run_name=RUN_NAME_OPTUNE_LGBM, experiment_id=experiment_id) as run:
    run_id_lgbm = run.info.run_id

In [ ]:
STUDY_NAME_LGBM = "LGBMClassifier_optuna_als"

mlflc_lgbm = MLflowCallback(
    tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
    metric_name="NDCG",
    create_experiment=False,
    mlflow_kwargs={'experiment_id': experiment_id, 'tags': {MLFLOW_PARENT_RUN_ID: run_id_lgbm}}
)

In [ ]:
study_lgbm = optuna.create_study(direction='maximize', 
                                 sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                                 study_name=STUDY_NAME_LGBM,
                                 storage=STUDY_DB_NAME,
                                 load_if_exists=False)

study_lgbm.optimize(objective_lgbm, 
                    n_trials=20, 
                    callbacks=[mlflc_lgbm]
                   )

best_params_lgbm = study_lgbm.best_params
print(f"Number of finished trials: {len(study_lgbm.trials)}")
print(f"Best params LGBM: {best_params_lgbm}")

In [ ]:
pipeline_lgbm_best = Pipeline([
    ('preprocessing', ColumnTransformer([
        ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
    ], remainder='passthrough')),
    ('model', LGBMClassifier(
        random_state=RANDOM_STATE, 
        verbose=-1,
        class_weight='balanced'
    ))
])

pipeline_lgbm_best.set_params(**study_lgbm.best_trial.user_attrs['pipeline_params'])
pipeline_lgbm_best.fit(X_train, y_train)

y_pred_proba_lgbm = pipeline_lgbm_best.predict_proba(X_test)

df_test_lgbm = df.loc[X_test.index].copy()
df_test_lgbm['y_pred_proba'] = y_pred_proba_lgbm[:, 1]

ndcg_lgbm_als, precision_lgbm_als, recall_lgbm_als, f1_lgbm_als = calculate_metrics(df_test_lgbm)
metrics_lgbm_als = {
    'NDCG': ndcg_lgbm_als,
    'Precision': precision_lgbm_als,
    'Recall': recall_lgbm_als,
    'F1': f1_lgbm_als
}

In [ ]:
RUN_NAME_LGBM = "best_optuna_lgbm_als"
REGISTRY_MODEL_NAME_LGBM = "best_optuna_lgbm_als"

signature = mlflow.models.infer_signature(X_test, y_test)
input_example = X_test[:10]
code_paths = ["ML_Experiments.ipynb"]

with mlflow.start_run(run_name=RUN_NAME_LGBM, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    
    lgbm_info = mlflow.sklearn.log_model(sk_model=pipeline_lgbm_best, 
                                        artifact_path='best_optuna_lgbm_als',
                                        registered_model_name=REGISTRY_MODEL_NAME_LGBM,
                                        input_example=input_example,
                                        code_paths=code_paths,
                                        await_registration_for=60
                                       )
    mlflow.log_metrics(metrics_lgbm_als)
    mlflow.log_params(best_params_lgbm)

## CatBoostClassifier

In [ ]:
def objective_catboost(trial: optuna.Trial) -> float:
    params = {
        'model__iterations': trial.suggest_int('iterations', 100, 1000, step=50),
        'model__depth': trial.suggest_int('depth', 3, 10),
        'model__learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'model__l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10, log=True)
    }
    
    pipeline_catboost = Pipeline([
        ('preprocessing', ColumnTransformer([
            ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
        ], remainder='passthrough')),
        ('model', CatBoostClassifier(
            random_state=RANDOM_STATE, 
            verbose=0, 
            auto_class_weights='Balanced'
        ))
    ])
    
    pipeline_catboost.set_params(**params)
    
    kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    
    for train_idx, val_idx in kfold.split(X_train, y_train):
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        pipeline_catboost.fit(X_fold_train, y_fold_train)
        y_pred_proba = pipeline_catboost.predict_proba(X_fold_val)
        
        df_val = df.loc[X_fold_val.index].copy()
        df_val['y_pred_proba'] = y_pred_proba[:, 1]
        
        ndcg, _, _, _ = calculate_metrics(df_val)
        
        trial.set_user_attr('pipeline_params', params)
    
    return ndcg

In [ ]:
RUN_NAME_OPTUNE_CATBOOST = 'CatBoostClassifier_optuna_als'

with mlflow.start_run(run_name=RUN_NAME_OPTUNE_CATBOOST, experiment_id=experiment_id) as run:
    run_id_catboost = run.info.run_id

In [ ]:
STUDY_NAME_CATBOOST = "CatBoostClassifier_optuna_als"

mlflc_catboost = MLflowCallback(
    tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
    metric_name="NDCG",
    create_experiment=False,
    mlflow_kwargs={'experiment_id': experiment_id, 'tags': {MLFLOW_PARENT_RUN_ID: run_id_catboost}}
)

In [ ]:
study_catboost = optuna.create_study(direction='maximize', 
                                     sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                                     study_name=STUDY_NAME_CATBOOST,
                                     storage=STUDY_DB_NAME,
                                     load_if_exists=True)

study_catboost.optimize(objective_catboost, 
                        n_trials=20, 
                        callbacks=[mlflc_catboost]
                       )

best_params_catboost = study_catboost.best_params
print(f"Number of finished trials: {len(study_catboost.trials)}")
print(f"Best params CatBoost: {best_params_catboost}")

In [ ]:
pipeline_catboost_best = Pipeline([
    ('preprocessing', ColumnTransformer([
        ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
    ], remainder='passthrough')),
    ('model', CatBoostClassifier(
        random_state=RANDOM_STATE, 
        verbose=0, 
        auto_class_weights='Balanced'
    ))
])

pipeline_catboost_best.set_params(**study_catboost.best_trial.user_attrs['pipeline_params'])
pipeline_catboost_best.fit(X_train, y_train)

y_pred_proba_catboost = pipeline_catboost_best.predict_proba(X_test)

df_test_catboost = df.loc[X_test.index].copy()
df_test_catboost['y_pred_proba'] = y_pred_proba_catboost[:, 1]

ndcg_catboost_als, precision_catboost_als, recall_catboost_als, f1_catboost_als = calculate_metrics(df_test_catboost)
metrics_catboost_als = {
    'NDCG': ndcg_catboost_als,
    'Precision': precision_catboost_als,
    'Recall': recall_catboost_als,
    'F1': f1_catboost_als
}

In [ ]:
Средний NDCG: 0.7814
Средний Precision: 0.6823
Средний Recall: 0.7589
Средний F1-Score: 0.7043

In [ ]:
RUN_NAME_CATBOOST = "best_optuna_catboost_als"
REGISTRY_MODEL_NAME_CATBOOST = "best_optuna_catboost_als"

signature = mlflow.models.infer_signature(X_test, y_test)
input_example = X_test[:10]
code_paths = ["ML_Experiments.ipynb"]

with mlflow.start_run(run_name=RUN_NAME_CATBOOST, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    
    catboost_info = mlflow.sklearn.log_model(sk_model=pipeline_catboost_best, 
                                             artifact_path='best_optuna_catboost_als',
                                             registered_model_name=REGISTRY_MODEL_NAME_CATBOOST,
                                             input_example=input_example,
                                             code_paths=code_paths,
                                             await_registration_for=60
                                            )
    mlflow.log_metrics(metrics_catboost_als)
    mlflow.log_params(best_params_catboost)

## Model comparison

In [ ]:
models_comparison = pd.DataFrame({
    'Model': ['LogisticRegression', 'RandomForest', 'XGBoost', 'LightGBM', 'CatBoost', 'XGBoost + ALS', 'LightGBM + ALS', 'CatBoost + ALS'],
    'NDCG': [ndcg_lr, ndcg_rf, ndcg_xgb, ndcg_lgbm, ndcg_catboost, ndcg_xgb_als, ndcg_lgbm_als, ndcg_catboost_als],
    'Precision': [precision_lr, precision_rf, precision_xgb, precision_lgbm, precision_catboost, precision_xgb_als, precision_lgbm_als, precision_catboost_als],
    'Recall': [recall_lr, recall_rf, recall_xgb, recall_lgbm, recall_catboost, recall_xgb_als, recall_lgbm_als, recall_catboost_als],
    'F1': [f1_lr, f1_rf, f1_xgb, f1_lgbm, f1_catboost, f1_xgb_als, f1_lgbm_als, f1_catboost_als]
})

models_comparison

In [ ]:
Model	NDCG	Precision	Recall	F1
0	LogisticRegression	0.751349	0.637015	0.596379	0.599915
1	RandomForest	0.775653	0.660585	0.742407	0.682372
2	XGBoost	0.778059	0.698832	0.734174	0.703595
3	LightGBM	0.778390	0.669718	0.750699	0.691776
4	CatBoost	0.778681	0.678187	0.747071	0.695479
5	XGBoost + ALS	0.781090	0.669943	0.763557	0.698493
6	LightGBM + ALS	0.781279	0.704709	0.749976	0.714586
7	CatBoost + ALS	0.780736	0.669796	0.762115	0.697661


In [ ]:
best_model_idx = models_comparison['NDCG'].idxmax()
print(f"\nЛучшая модель: {models_comparison.iloc[best_model_idx]['Model']} с NDCG = {models_comparison.iloc[best_model_idx]['NDCG']:.4f}")

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Сохраним обученный итоговый пайплайн.

</div>

In [ ]:
MODEL_NAME = 'pipeline_lgb_best.pkl'
with open(MODEL_NAME, 'wb') as file:
    pickle.dump(pipeline_lgbm_best, file)

In [ ]:
def show_vacancy_predictions(pipeline, X_test, y_test, df,
                             n_top=10, vacancy_id=None, random_state=None):
    """
    Показывает предсказания пайплайна для выбранной вакансии из тест-сета.

    Args:
        pipeline     : обученный sklearn Pipeline
        X_test       : тестовые признаки (те же колонки, на которых обучался pipeline)
        y_test       : истинные метки
        df           : исходный датафрейм со всеми колонками (для отображения)
        n_top        : сколько топ-резюме показать
        vacancy_id   : None = случайная вакансия; иначе конкретный ID
        random_state : seed для воспроизводимости случайного выбора

    Returns:
        pd.DataFrame, отсортированный по pred_proba убыванием
    """
    df_test = df.loc[X_test.index].copy()
    df_test['target'] = y_test.values

    if vacancy_id is None:
        rng = np.random.RandomState(random_state)
        vacancy_id = rng.choice(df_test['vacancy_id'].unique())

    mask = df_test['vacancy_id'] == vacancy_id
    if not mask.any():
        print(f"[!] vacancy_id={vacancy_id} не найден в тестовой выборке.")
        return None

    df_vac = df_test[mask].copy()
    X_vac  = X_test.loc[df_vac.index]

    df_vac['pred_proba'] = pipeline.predict_proba(X_vac)[:, 1]
    df_vac = df_vac.sort_values('pred_proba', ascending=False).reset_index(drop=True)

    r0       = df_vac.iloc[0]
    vac_name = str(r0.get('vacancy_name', '—'))
    vac_desc = str(r0.get('vacancy_description', '—'))
    SEP, SEP2 = "=" * 90, "─" * 90

    print(SEP)
    print(f"  ВАКАНСИЯ : {vac_name}   [id={vacancy_id}]")
    print(f"  Опыт     : {r0.get('vacancy_experience','—')}  |  "
          f"Занятость : {r0.get('vacancy_employment','—')}  |  "
          f"График : {r0.get('vacancy_schedule','—')}")
    print(SEP2)
    print("  ОПИСАНИЕ ВАКАНСИИ:")
    for line in vac_desc[:1200].split('\n')[:25]:
        if line.strip():
            print(f"    {line.strip()}")
    if len(vac_desc) > 1200:
        print("    [... сокращено]")
    print(SEP)

    n_pos = int(df_vac['target'].sum())
    print(f"\n  ТОП-{n_top} из {len(df_vac)} кандидатов  "
          f"(всего реально подходящих в выборке: {n_pos})\n")

    for rank, (_, row) in enumerate(df_vac.head(n_top).iterrows(), 1):
        icon   = "✅" if row['target'] == 1 else "❌"
        exp    = str(row.get('resume_last_experience_description', '—'))
        skills = str(row.get('resume_skills', '—'))
        print(SEP2)
        print(f"  #{rank}  {icon}  score={row['pred_proba']:.4f}  "
              f"target={int(row['target'])}  [resume_id={row.get('resume_id','—')}]")
        print(f"  Должность : {row.get('resume_last_position','—')}")
        print(f"  Локация   : {row.get('resume_location','—')}  |  "
              f"Опыт: {row.get('resume_experience_months','—')} мес.")
        print(f"  Навыки    : {skills[:180]}{'...' if len(skills) > 180 else ''}")
        print(f"  Описание последнего места:")
        for line in exp[:700].split('\n')[:14]:
            if line.strip():
                print(f"    {line.strip()}")
        if len(exp) > 700:
            print("    [... сокращено]")
        print()

    print(SEP2)
    n_hit = int(df_vac.head(n_top)['target'].sum())
    print(f"\n  Попаданий в топ-{n_top}: {n_hit}/{n_top} ({100*n_hit//n_top}%)")
    print(f"  Всего релевантных в тест-сете для этой вакансии: {n_pos}\n")

    return df_vac

In [ ]:
# Случайная вакансия из тест-сета, лучший пайплайн
result = show_vacancy_predictions(pipeline_lgbm_best, X_test, y_test, df,
                                  n_top=10, vacancy_id=126372900)

# Зафиксировать вакансию и сравнить другой пайплайн:
# vacancy_id_fixed = result.iloc[0]['vacancy_id']
# show_vacancy_predictions(pipeline_lgbm_als, X_test, y_test, df,
#                          n_top=10, vacancy_id=vacancy_id_fixed)